In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split

from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [3]:
TRAIN_DIR = "/content/drive/MyDrive/disaster_final/train"
VAL_DIR = "/content/drive/MyDrive/disaster_final/validation"
TEST_DIR = "/content/drive/MyDrive/disaster_final/test"

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 10

In [4]:
DISASTER_TYPES = ["earthquake","flood","wildfire"]
SEVERITY_LEVELS = ["high","low","medium"]

TYPE_LABEL = {

    "earthquake":0,
    "flood":1,
    "wildfire":2
}

In [5]:
def load_paths(dataset_dir):

    paths=[]
    labels=[]

    for disaster in DISASTER_TYPES:
        for severity in SEVERITY_LEVELS:

            folder=os.path.join(dataset_dir,disaster,severity)

            if not os.path.exists(folder):
                continue

            for file in os.listdir(folder):

                if file.lower().endswith((".jpg",".jpeg",".png")):

                    paths.append(os.path.join(folder,file))
                    labels.append(TYPE_LABEL[disaster])

    return np.array(paths),np.array(labels)

In [6]:
train_paths,train_labels = load_paths(TRAIN_DIR)
val_paths,val_labels = load_paths(VAL_DIR)
test_paths,test_labels = load_paths(TEST_DIR)

print("Train images:",len(train_paths))
print("Validation images:",len(val_paths))
print("Test images:",len(test_paths))

Train images: 2244
Validation images: 892
Test images: 462


In [7]:
def load_image(path,label):

    img=tf.io.read_file(path)

    img=tf.io.decode_image(img,channels=3)

    img.set_shape([None,None,3])

    img=tf.image.resize(img,[IMG_SIZE,IMG_SIZE])

    img=tf.cast(img,tf.float32)/255.0

    return img,label

In [8]:
train_ds=tf.data.Dataset.from_tensor_slices((train_paths,train_labels))

train_ds=train_ds.map(load_image,
                      num_parallel_calls=tf.data.AUTOTUNE)

train_ds=train_ds.shuffle(2000)

train_ds=train_ds.batch(BATCH_SIZE)

train_ds=train_ds.prefetch(tf.data.AUTOTUNE)


val_ds=tf.data.Dataset.from_tensor_slices((val_paths,val_labels))

val_ds=val_ds.map(load_image,
                  num_parallel_calls=tf.data.AUTOTUNE)

val_ds=val_ds.batch(BATCH_SIZE)

In [9]:
backbone = load_model("/content/drive/MyDrive/Backbone_model_V-Zeus.keras.keras")

print("Backbone loaded")

Backbone loaded


In [10]:
for layer in backbone.layers[1].layers[:30]:
    layer.trainable=False

In [11]:
x = backbone.output

x = Dense(256,activation="relu")(x)

x = Dropout(0.5)(x)

output = Dense(3,activation="softmax")(x)

model_stage2 = Model(backbone.input,output)

model_stage2.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,378,278 (16.70 MB)

 Trainable params: 4,330,227 (16.52 MB)

 Non-trainable params: 48,051 (187.70 KB)

In [12]:
model_stage2.compile(
    optimizer=Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [13]:
model_stage2.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Epoch 1/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 720s 2s/step - accuracy: 0.4438 - loss: 1.0650 - val_accuracy: 0.3879 - val_loss: 1.4401
Epoch 2/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 48s 163ms/step - accuracy: 0.7402 - loss: 0.7090 - val_accuracy: 0.8240 - val_loss: 0.5255
Epoch 3/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 55s 225ms/step - accuracy: 0.8245 - loss: 0.5192 - val_accuracy: 0.8969 - val_loss: 0.3390
Epoch 4/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 73s 168ms/step - accuracy: 0.8520 - loss: 0.4451 - val_accuracy: 0.9182 - val_loss: 0.2720
Epoch 5/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 55s 220ms/step - accuracy: 0.8869 - loss: 0.3610 - val_accuracy: 0.9249 - val_loss: 0.2253
Epoch 6/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 47s 165ms/step - accuracy: 0.9013 - loss: 0.3122 - val_accuracy: 0.9339 - val_loss: 0.1963
Epoch 7/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 82s 162ms/step - accuracy: 0.9109 - loss: 0.2591 - val_accuracy: 0.9395 - val_loss: 0.1744
Epoch 8/10
141/141 ━━━━━━━━━━━━━━━━━━━━ 45s 149ms/step - accuracy: 0.9269 - loss: 0.2

In [14]:
test_ds=tf.data.Dataset.from_tensor_slices((test_paths,test_labels))

test_ds=test_ds.map(load_image).batch(BATCH_SIZE)

model_stage2.evaluate(test_ds)

29/29 ━━━━━━━━━━━━━━━━━━━━ 105s 4s/step - accuracy: 0.9305 - loss: 0.2118


[0.2747233808040619, 0.9090909361839294]

In [15]:
model_stage2.save("/content/drive/MyDrive/classifier_model_V_Hera.keras")

print("Classifier model version-Hera saved to Drive!")

Classifier model version-Hera saved to Drive!
